# 题材来源与周期样本核验

2026-09-22。复算本次研究，不修改产品或数据库。来源包括本机归档、已配置 Tushare 兼容通道的东财分类与行情、公开官方名称核对。

主要结论窗口 2026-08-24—09-22，统计沪深非 ST 主板；历史成分按请求日期使用，但没有历史发布时间证明。概念归属不代表当日上涨原因。早于 08-24 的部分收益缺失，不用于反馈阈值结论。不是收益回测。

原始输入已按复算需要裁剪行情行，目录及历史成分保留查询结果；该裁剪不改变主要比较期的前日队列反馈覆盖。

In [ ]:
# 只读来源查询（定义供审阅，不自动联网，不打印配置或凭据）
def refresh_source_example():
    from shilun.api.review import _open_store
    from shilun.data.providers.tushare_provider import TushareConfig, TushareDailyClient
    client = TushareDailyClient(TushareConfig.from_env())
    catalog = client._call_api('dc_index', trade_date='20260921')
    members = client._call_api('dc_member', ts_code='BK0825.DC', trade_date='20260921')
    history = client._call_api('dc_member', ts_code='BK0825.DC', start_date='20260824', end_date='20260922')
    older_history = client._call_api('dc_member', ts_code='BK0825.DC', start_date='20260803', end_date='20260823')
    daily = client._call_api('daily', trade_date='20260831')
    missing_pool = client._call_api('limit_list_d', trade_date='20260909', limit_type='U')
    store = _open_store()
    try:
        archived = list(store.db.review_board_pools.find({'date': {'$gte': '2026-08-01', '$lte': '2026-09-22'}}, {'_id': 0}))
    finally:
        store.close()
    return catalog, members, history, older_history, daily, missing_pool, archived


In [ ]:
import json,collections,statistics
from pathlib import Path
base = Path('research/evidence/theme_cycle_20260922')
if not base.exists(): base = Path('evidence/theme_cycle_20260922')
load=lambda n:json.loads((base/n).read_text())
p=load('theme_pools_20260922.json'); b=load('theme_bars_20260922.json'); h=load('theme_dc_history_20260922.json')
pools={(r['date'],r['kind']):r for r in p['pools']}
for r in load('theme_extra_pools_20260922.json'): pools[(r['date'],r['kind'])]=r
sessions=sorted(r['cal_date'][:4]+'-'+r['cal_date'][4:6]+'-'+r['cal_date'][6:] for r in b['calendar'] if r['is_open']==1)
bars={(r['date'],r['ticker']):r for r in b['bars']}
if (base/'theme_daily_api_20260922.json').exists():
 for r in load('theme_daily_api_20260922.json'):
  dt=r['trade_date'];d=f'{dt[:4]}-{dt[4:6]}-{dt[6:]}';t=r['ts_code'];bars[(d,t)]={**bars.get((d,t),{}),**r,'date':d,'ticker':t,'volume':r.get('vol')}
focus={'国芳集团':'601086.SH','金健米业':'600127.SH','闽东电力':'000993.SZ','哈药股份':'600664.SH','百花医药':'600721.SH','开开实业':'600272.SH'}
# Common population: non-ST Shanghai/Shenzhen main board, avoiding varying 20/30% universes.
def eligible(r): return r['ticker'].startswith(('60','000','001','002','003')) and 'ST' not in r.get('name','').upper()
def up(d): return {r['ticker']:r for r in pools.get((d,'up'),{}).get('stocks',[]) if eligible(r)}
def br(d): return {r['ticker']:r for r in pools.get((d,'broken'),{}).get('stocks',[]) if eligible(r)}
def pct(d,t):
 r=bars.get((d,t),{}); v=r.get('pre_close'); return 100*(r['close']/v-1) if v and r.get('close') else None
names={r['ticker']:r['name'] for d in pools.values() for r in d['stocks']}
rows=[]
for code,core in [('BK1106.DC','600721.SH'),('BK0899.DC','600721.SH'),('BK0825.DC','601086.SH')]:
 md=collections.defaultdict(set)
 for r in h[code]:md[r['trade_date']].add(r['con_code'])
 for i,d in enumerate(sessions):
  if d<'2026-08-03': continue
  todayset=md[d.replace('-','')]; u=up(d); bb=br(d); prev=sessions[i-1] if i else ''
  # Frozen previous cohort uses previous-date membership, does not delete today's failures.
  prior=set(up(prev))&md[prev.replace('-','')]
  vals=[pct(d,t) for t in prior if pct(d,t) is not None]
  active=set(u)&todayset; broken=set(bb)&todayset
  followers=set(); activitydays=0
  for q in sessions[max(0,i-2):i+1]:
   fs=set(up(q))&md[q.replace('-','')]-{core}; followers|=fs; activitydays+=bool(fs)
  rows.append(dict(code=code,date=d,members=len(todayset),up=len(active),broken=len(broken),core_up=core in active,followers3=len(followers),active_days3=activitydays,promotion_num=len(prior&set(u)),promotion_den=len(prior),feedback_median=round(statistics.median(vals),2) if vals else None,feedback_coverage=f'{len(vals)}/{len(prior)}',height=max([u[t]['boards'] or 1 for t in active],default=0),stocks=[f'{names[t]}:{u[t]["boards"]}' for t in sorted(active)],broken_stocks=[names[t] for t in sorted(broken)]))
market=[]
for i,d in enumerate(sessions):
 if d<'2026-08-03':continue
 u=up(d);z=br(d);prev=up(sessions[i-1]) if i else {};v=[pct(d,t) for t in prev if pct(d,t) is not None]
 market.append(dict(date=d,up=len(u),broken=len(z),broken_rate=round(100*len(z)/(len(u)+len(z)),1) if len(u)+len(z) else None,height=max([r['boards'] or 1 for r in u.values()],default=0),feedback=round(statistics.median(v),2) if v else None))
focusrows=[]
for name,t in focus.items():
 for d in sessions:
  r=bars.get((d,t),{});u=up(d)
  focusrows.append(dict(name=name,ticker=t,date=d,up=t in u,boards=u.get(t,{}).get('boards'),pct=round(pct(d,t),2) if pct(d,t) is not None else None,**{k:r.get(k) for k in ['close','open','low','high','pre_close','volume','amount']}))
result={'scope':'non-ST SH/SZ main board, dated DC membership; archived/retrieved at audit time, not availability-certified as-of replay','rows':rows,'market':market,'focus':focusrows,'missing_pools':[(d,k) for d in sessions for k in ['up','broken'] if (d,k) not in pools], 'member_days':{k:len(set(r['trade_date'] for r in v)) for k,v in h.items()},'bar_duplicate_count':len(b['bars'])-len(bars)}
(base/'theme_audit_result_20260922.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))

print('主题日期行数:', len(result['rows']))
print('缺板池:', result['missing_pools'])
print('比较期反馈缺口:', [(r['date'], r['code'], r['feedback_coverage']) for r in result['rows'] if r['date'] >= '2026-08-24' and len(set(r['feedback_coverage'].split('/'))) > 1])


## 阈值敏感性

这是简化退潮信号的 18 组局部对照，不是完整主线算法。不能把无触发等同于无风险；完整概念涨停可能来自别的原因，正式规则还需固定本轮参与者。

In [ ]:
import itertools
sensitivity=[]
for core, code, start in [('百花医药','BK1106.DC','2026-08-04'),('国芳集团','BK0825.DC','2026-08-28')]:
    cr={r['date']:r for r in result['focus'] if r['name']==core and r['date']>=start}
    tr={r['date']:r for r in result['rows'] if r['code']==code}
    for threshold, confirm in itertools.product([10,12,15],[1,2,3]):
        peak=0; streak=0; first=None
        for d in sorted(cr):
            peak=max(peak,cr[d]['close'])
            weak=100*(1-cr[d]['close']/peak)>=threshold and tr[d]['up']<=1
            streak=streak+1 if weak else 0
            if streak>=confirm and first is None: first=d
        sensitivity.append({'core':core,'drawdown_pct':threshold,'confirm_days':confirm,'first_exit_signal':first})
print(json.dumps(sensitivity,ensure_ascii=False,indent=2))
